# TabFM exploratory benchmark

This notebook adds **TabFM** as a separate exploratory benchmark for the ACLED protest diffusion project.

> We include TabFM as an additional predictive benchmark to test whether a tabular foundation model can improve out-of-time protest-risk ranking. The substantive hypothesis tests remain based on interpretable logit specifications and the established feature comparisons.

This notebook is intentionally isolated from the main pipeline. It reads the existing 100 km cell-month panel and writes any TabFM outputs only inside `TabFMrun/outputs/`.

## How to use this notebook

- Run this in **Google Colab**.
- The notebook installs TabFM in the Colab runtime.
- It clones the GitHub repository if the project files are not already present.
- The default run is deliberately conservative: it uses a sampled training context for TabFM and evaluates on the out-of-time 2016-2020 test period.
- The main logit/random forest notebook remains the source for substantive H1-H3 interpretation.

In [ ]:
# Colab/runtime configuration
import os
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Mat99999/acled-spatiotemporal-conflict-model.git"
REPO_DIRNAME = "acled-spatiotemporal-conflict-model"

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if sys.version_info < (3, 11):
    raise RuntimeError(
        f"TabFM currently requires Python >= 3.11. This runtime is {sys.version.split()[0]}."
    )

if IN_COLAB:
    PROJECT_DIR = Path("/content") / REPO_DIRNAME
    if not PROJECT_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
else:
    cwd = Path.cwd().resolve()
    PROJECT_DIR = cwd.parent if cwd.name == "TabFMrun" else cwd

os.chdir(PROJECT_DIR)

TABFM_DIR = PROJECT_DIR / "TabFMrun"
OUTPUT_DIR = TABFM_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PANEL_PATH = PROJECT_DIR / "outputs" / "panels" / "panel_100km_month.csv"
MAIN_RESULTS_PATH = PROJECT_DIR / "outputs" / "tables" / "04_model_results_100km.csv"

print("IN_COLAB:", IN_COLAB)
print("PROJECT_DIR:", PROJECT_DIR)
print("PANEL_PATH exists:", PANEL_PATH.exists())
print("OUTPUT_DIR:", OUTPUT_DIR)

In [ ]:
# Install TabFM. This cell is Colab-first and may take several minutes.
# If the TabFM package changes, check: https://github.com/google-research/tabfm

INSTALL_TABFM = True

if INSTALL_TABFM:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/google-research/tabfm.git#egg=tabfm[pytorch]"],
        check=True,
    )

print("TabFM installation cell completed.")

## Experimental design

The main project uses interpretable logit specifications and random forest benchmarks. This notebook keeps that design logic but treats TabFM only as an extra predictive benchmark.

The key choices are:

- **Main grid:** 100 x 100 km.
- **Training context:** pre-2016 rows, sampled to keep Colab runtime manageable.
- **Test period:** 2016-2020 out-of-time rows.
- **Default feature sets:** local history and local plus neighbors.
- **Main comparison metric:** average precision and top-risk precision, because protest cell-months are rare.

In [ ]:
# User-tunable settings

RANDOM_STATE = 42

# TabFM reads training examples as context. Larger values may improve performance but use more memory.
MAX_CONTEXT_ROWS = 50_000

# Keep all positive training cases where possible, then sample negatives around them.
NEGATIVE_TO_POSITIVE_RATIO = 5

# Evaluate the full out-of-time test period by default. Reduce if Colab memory is limited.
MAX_TEST_ROWS = None

# Prediction batching reduces memory pressure.
PREDICTION_BATCH_SIZE = 2_048

# Start with the feature sets most relevant to H1. Add D_fatality_split if runtime is acceptable.
FEATURE_SETS_TO_RUN = [
    "B_local_history",
    "C_local_plus_neighbors",
    "D_fatality_split",
]

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, brier_score_loss, precision_recall_fscore_support, roc_auc_score

if not PANEL_PATH.exists():
    raise FileNotFoundError(f"Could not find panel file: {PANEL_PATH}")

panel = pd.read_csv(PANEL_PATH, parse_dates=["month"])

target_col = "target_protest_next_month"
panel[target_col] = panel[target_col].astype(int)

train_df = panel.loc[panel["year"] <= 2015].copy()
test_df = panel.loc[(panel["year"] >= 2016) & (panel["year"] <= 2020)].copy()

if MAX_TEST_ROWS is not None and len(test_df) > MAX_TEST_ROWS:
    test_df = test_df.sample(n=MAX_TEST_ROWS, random_state=RANDOM_STATE).sort_values(["month", "cell_id"])

print("Panel rows:", len(panel))
print("Train context candidate rows:", len(train_df), "positive share:", train_df[target_col].mean().round(4))
print("Out-of-time test rows:", len(test_df), "positive share:", test_df[target_col].mean().round(4))

In [ ]:
# Feature sets. These are designed for the TabFM benchmark and mirror the logic of the main models.
# The main notebook remains the authoritative source for the hypothesis-test specifications.

A_PLACE_TIME = ["grid_x", "grid_y", "year", "sin_month", "cos_month"]

B_LOCAL_HISTORY = A_PLACE_TIME + [
    "events_lag_1", "protest_lag_1",
    "events_lag_2", "protest_lag_2",
    "events_lag_6", "protest_lag_6",
    "events_lag_12", "protest_lag_12",
    "events_roll_6", "events_roll_12",
]

C_LOCAL_PLUS_NEIGHBORS = B_LOCAL_HISTORY + [
    "neighbor_protests",
    "neighbor_protest_cells",
]

D_FATALITY_SPLIT = C_LOCAL_PLUS_NEIGHBORS + [
    "neighbor_fatal_protests",
    "neighbor_nonfatal_protests",
]

FEATURE_SETS = {
    "B_local_history": B_LOCAL_HISTORY,
    "C_local_plus_neighbors": C_LOCAL_PLUS_NEIGHBORS,
    "D_fatality_split": D_FATALITY_SPLIT,
}

for name in FEATURE_SETS_TO_RUN:
    missing = [c for c in FEATURE_SETS[name] if c not in panel.columns]
    if missing:
        raise ValueError(f"Missing columns for {name}: {missing}")

print({name: len(FEATURE_SETS[name]) for name in FEATURE_SETS_TO_RUN})

In [ ]:
def build_context_sample(df, target, max_rows, neg_pos_ratio, random_state):
    positives = df.loc[df[target] == 1]
    negatives = df.loc[df[target] == 0]

    max_negatives = min(len(negatives), len(positives) * neg_pos_ratio)
    sampled_negatives = negatives.sample(n=max_negatives, random_state=random_state)
    context = pd.concat([positives, sampled_negatives], axis=0)

    if len(context) > max_rows:
        pos_keep = min(len(positives), max_rows // (neg_pos_ratio + 1))
        neg_keep = max_rows - pos_keep
        positives_small = positives.sample(n=pos_keep, random_state=random_state) if len(positives) > pos_keep else positives
        negatives_small = negatives.sample(n=neg_keep, random_state=random_state)
        context = pd.concat([positives_small, negatives_small], axis=0)

    return context.sample(frac=1.0, random_state=random_state).reset_index(drop=True)


context_df = build_context_sample(
    train_df,
    target=target_col,
    max_rows=MAX_CONTEXT_ROWS,
    neg_pos_ratio=NEGATIVE_TO_POSITIVE_RATIO,
    random_state=RANDOM_STATE,
)

print("TabFM context rows:", len(context_df))
print("TabFM context positive share:", round(context_df[target_col].mean(), 4))

In [ ]:
from tabfm import TabFMClassifier
from tabfm import tabfm_v1_0_0_pytorch as tabfm_v1_0_0

tabfm_model = tabfm_v1_0_0.load()

print("Loaded TabFM PyTorch backend.")

In [ ]:
def predict_proba_batched(clf, X, batch_size):
    parts = []
    for start in range(0, len(X), batch_size):
        stop = min(start + batch_size, len(X))
        batch = X.iloc[start:stop]
        prob = clf.predict_proba(batch)
        class_index = list(clf.classes_).index(1)
        parts.append(prob[:, class_index])
        print(f"Predicted rows {start:,} to {stop:,} of {len(X):,}")
    return np.concatenate(parts)


def top_k_table(y_true, y_score, shares=(0.01, 0.05, 0.10)):
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    order = np.argsort(-y_score)
    base_rate = y_true.mean()
    rows = []
    for share in shares:
        n = max(1, int(np.ceil(len(y_true) * share)))
        idx = order[:n]
        precision = y_true[idx].mean()
        recall = y_true[idx].sum() / max(1, y_true.sum())
        lift = precision / base_rate if base_rate > 0 else np.nan
        rows.append({
            "top_share": share,
            "n_flagged": n,
            "precision_at_top": precision,
            "recall_at_top": recall,
            "lift_vs_base_rate": lift,
        })
    return pd.DataFrame(rows)


all_metrics = []
all_topk = []

for feature_set_name in FEATURE_SETS_TO_RUN:
    features = FEATURE_SETS[feature_set_name]
    print("\n=== TabFM feature set:", feature_set_name, "===")

    X_context = context_df[features].copy()
    y_context = context_df[target_col].to_numpy(dtype=int)
    X_test = test_df[features].copy()
    y_test = test_df[target_col].to_numpy(dtype=int)

    clf = TabFMClassifier(model=tabfm_model)
    clf.fit(X_context, y_context)

    y_score = predict_proba_batched(clf, X_test, PREDICTION_BATCH_SIZE)

    threshold = y_test.mean()
    y_pred = (y_score >= threshold).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test, y_pred, average="binary", zero_division=0
    )

    metrics = {
        "model": f"TabFM_{feature_set_name}",
        "feature_set": feature_set_name,
        "n_features": len(features),
        "context_rows": len(X_context),
        "test_rows": len(X_test),
        "test_positive_share": y_test.mean(),
        "roc_auc": roc_auc_score(y_test, y_score),
        "avg_precision": average_precision_score(y_test, y_score),
        "brier": brier_score_loss(y_test, y_score),
        "threshold_base_rate": threshold,
        "precision_at_base_rate_threshold": precision,
        "recall_at_base_rate_threshold": recall,
        "f1_at_base_rate_threshold": f1,
    }
    all_metrics.append(metrics)

    pred_out = test_df[["cell_id", "grid_x", "grid_y", "month", target_col]].copy()
    pred_out["tabfm_score"] = y_score
    pred_path = OUTPUT_DIR / f"tabfm_predictions_100km_{feature_set_name}.csv"
    pred_out.to_csv(pred_path, index=False)
    print("Saved predictions:", pred_path)

    tk = top_k_table(y_test, y_score)
    tk.insert(0, "model", f"TabFM_{feature_set_name}")
    tk.insert(1, "feature_set", feature_set_name)
    all_topk.append(tk)

metrics_df = pd.DataFrame(all_metrics).sort_values("avg_precision", ascending=False)
topk_df = pd.concat(all_topk, ignore_index=True)

metrics_path = OUTPUT_DIR / "tabfm_metrics_100km.csv"
topk_path = OUTPUT_DIR / "tabfm_precision_at_top_k_100km.csv"
metrics_df.to_csv(metrics_path, index=False)
topk_df.to_csv(topk_path, index=False)

display(metrics_df)
display(topk_df)
print("Saved metrics:", metrics_path)
print("Saved top-k table:", topk_path)

In [ ]:
# Compare TabFM with the existing 100 km benchmark table.
# This is a predictive comparison only. It is not used as a hypothesis test.

if MAIN_RESULTS_PATH.exists():
    main_results = pd.read_csv(MAIN_RESULTS_PATH)
    main_compare = main_results[["model", "feature_set", "n_features", "roc_auc", "avg_precision", "brier", "f1"]].copy()
    main_compare["source"] = "main_pipeline"

    tabfm_compare = metrics_df[["model", "feature_set", "n_features", "roc_auc", "avg_precision", "brier", "f1_at_base_rate_threshold"]].copy()
    tabfm_compare = tabfm_compare.rename(columns={"f1_at_base_rate_threshold": "f1"})
    tabfm_compare["source"] = "TabFMrun"

    comparison = pd.concat([main_compare, tabfm_compare], ignore_index=True)
    comparison = comparison.sort_values("avg_precision", ascending=False)

    comparison_path = OUTPUT_DIR / "tabfm_vs_main_pipeline_100km.csv"
    comparison.to_csv(comparison_path, index=False)
    display(comparison)
    print("Saved comparison:", comparison_path)
else:
    print("Main results table not found, skipping comparison:", MAIN_RESULTS_PATH)

## How to interpret the result

Use TabFM as an additional predictive benchmark only.

A useful result would be something like:

- TabFM improves average precision or top-risk precision over the main random forest/logit models.
- TabFM performs similarly, which suggests the existing pipeline is already competitive.
- TabFM performs worse, which suggests the structured, domain-specific feature design remains important for this task.

Do **not** use this notebook to claim that H1, H2, or H3 is confirmed. Those substantive claims should still come from the interpretable logit specifications and the established feature comparisons in the main notebook.